In [1]:
import pandas as pd
import lightgbm as lgb
print(lgb.__version__)

4.7.0


In [ ]:
import joblib
from sklearn.metrics import recall_score, f1_score, classification_report

#### 1. data load

In [ ]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=["target"])
y_train = train["target"]
X_val = val.drop(columns=["target"])
y_val = val["target"]

#### 2. LightGBM training

In [ ]:
model = lgb.LGBMClassifier(
    objective="binary",
    class_weight="balanced",   # 클래스 불균형(32:68) 보정
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50)] # val set 성능이 더 이상 안 좋아지면 자동으로 멈춰서 과적합 방지
)

#### 3. 평가 (Recall, F1)

In [ ]:
y_pred = model.predict(X_val)

recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Recall: {recall:.4f}, F1: {f1:.4f}")
print(classification_report(y_val, y_pred))

#### 4. result save

In [ ]:
joblib.dump(model, "../models/lightgbm.joblib")

# feature importance
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)
importance_df.to_csv("../reports/lightgbm_importance.csv", index=False)